# Smart Light TinyML\nTrain and export the Dark/Normal/Bright model used by the ESP32 integration project.

In [ ]:
import numpy as np\nimport tensorflow as tf\n\nx=np.array([[.05],[.15],[.25],[.40],[.50],[.60],[.70],[.85],[.95]],dtype=np.float32)\ny=np.array([0,0,0,1,1,1,2,2,2],dtype=np.int32)\n\nmodel=tf.keras.Sequential([tf.keras.layers.Input(shape=(1,)),tf.keras.layers.Dense(8,activation='relu'),tf.keras.layers.Dense(3,activation='softmax')])\nmodel.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])\nmodel.fit(x,y,epochs=250,verbose=0)\n\nconverter=tf.lite.TFLiteConverter.from_keras_model(model)\ntflite_model=converter.convert()\nopen('light_level_classifier.tflite','wb').write(tflite_model)\n\ninterpreter=tf.lite.Interpreter(model_content=tflite_model)\ninterpreter.allocate_tensors()\ninp=interpreter.get_input_details()[0]; out=interpreter.get_output_details()[0]\nfor value in [.1,.5,.9]:\n    interpreter.set_tensor(inp['index'],np.array([[value]],dtype=np.float32))\n    interpreter.invoke()\n    pred=int(np.argmax(interpreter.get_tensor(out['index'])[0]))\n    print(value,['Dark','Normal','Bright'][pred])

Run `python convert_model.py` after generating the TFLite file to create the model header used by the ESP32.